# Unified Lab 5: The Actor-Critic Bridge & Continuous Control

## The Scenario
**You are an AI Architect at "AeroLogistics."** Your division is designing the thrust-stabilization system for a new fleet of heavy-lift delivery drones. The stabilization physics can be modeled mathematically as a classic inverted pendulum problem. Historically, the team used simple Tabular Q-Learning (Value-Based) methods, but these drones operate in environments with **continuous state spaces** (e.g., precise angles, velocities, and positions).

You must guide your AI coding assistant to upgrade the legacy system to a sophisticated Actor-Critic model, solving the severe limitations of state discretization and the high variance of basic policy gradients.

## Milestone 1 (The Baseline): The Discretization Trap
**Objective:** Establish a baseline model-free Q-Learning agent and observe the severe limitations of forcing continuous states into discrete tables, as well as the behavior of basic $\epsilon$-greedy exploration.


### The Architect's Blueprint

Do not ask the AI for generic Q-Learning. Construct a prompt instructing your AI to build a class-based Q-Learning agent that satisfies the following constraints:
* Use the `gymnasium` library to load the `CartPole-v1` environment.
* Because `CartPole` returns a continuous 4D array, the agent must include a method `discretize_state` to discretize (bucket) these continuous variables into a finite set of bins.
* Implement an Epsilon-Greedy exploration strategy with a decay rate.
* The class must be named `agent` and satisfy all assertions in the provided Test Harness.
* Train for 500 episodes and output a plot of the rolling average reward.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*


In [ ]:
# MILESTONE 1 TEST HARNESS - DO NOT MODIFY

import numpy as np

# Test 1: Ensure the agent has a discretization method
assert hasattr(agent, 'discretize_state'), "Agent must have a 'discretize_state' method."

# Test 2: Ensure discretization maps a 4D continuous array to a discrete tuple
sample_continuous_state = np.array([0.01, 0.5, -0.05, -0.1])
discrete_state = agent.discretize_state(sample_continuous_state)
assert isinstance(discrete_state, tuple), "Discretized state must be a tuple for Q-Table indexing."
assert len(discrete_state) == 4, "Discretized state must represent 4 dimensions."

# Test 3: Ensure Q-table is instantiated
assert hasattr(agent, 'q_table'), "Agent must have a 'q_table' attribute."

### Architect's Audit (Milestone 1)

1. Review the `discretize_state` helper function generated by your AI. What happens mathematically when the drone's physical angle is sitting exactly on the boundary line between two of your predefined buckets? How does this arbitrary bucketing impact the agent's ability to accurately perceive momentum (velocity)? Finally, look at your learning curve: how efficiently does the blind $\epsilon$-greedy strategy navigate this rigid, discretized space during early episodes?

ENTER YOUR ANSWERS HERE

## Milestone 2 (The Architecture Upgrade): Continuous States via Neural Networks

**Objective:** Replace the discrete Q-table with a neural network capable of directly processing continuous state spaces. You will build a Policy network (an "Actor") that evaluates the state and outputs action probabilities.

### The Architect's Blueprint

Instruct your AI assistant to build a PyTorch agent class (`ReinforceAgent`) that satisfies the Test Harness. Give the AI these specific constraints:
* **The Network:** Use `torch.nn` to create a neural network architecture.
* **The Policy (Actor):** It must take the 4D continuous state as input, pass it through hidden layers, and use a `Softmax` activation function to output a probability distribution over the 2 possible discrete actions (Left, Right).
* Instantiate the class as `reinforce_agent` so it can be evaluated by the Test Harness. *(Note: We are not writing the training loop yet, just engineering the forward pass).*

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 2 TEST HARNESS - DO NOT MODIFY

import torch
import torch.nn as nn

# Create a dummy continuous state tensor matching CartPole's 4D observation space
dummy_state = torch.randn(1, 4)

# Test 1: Verify the Policy network's output
action_probs = reinforce_agent.get_action_probs(dummy_state)
assert action_probs.shape == (1, 2), "Policy must output probabilities for the 2 possible actions."
assert torch.isclose(torch.sum(action_probs), torch.tensor(1.0)), "Outputs must be a valid probability distribution summing to 1."

### Architect's Audit (Milestone 2)

1. Pass a random test state into your newly generated Policy network and examine the output probabilities. Because the network has not been trained yet (the weights are randomly initialized), what specific probability distribution does the network output closely resemble? Mathematically, why is initializing the network this way highly advantageous for the agent's exploration during the very first few episodes, compared to a deterministic policy?

ENTER YOUR ANSWERS HERE

## Milestone 3 (The High Variance Trap): Vanilla REINFORCE Training Loop
**Objective:** Implement the pure Policy Gradient theorem (REINFORCE) to train your continuous Policy network. You will observe how relying on raw, full-episode returns ($G_t$) causes severe instability and variance during learning.

### The Architect's Blueprint

Instruct your AI assistant to add an `update` method to your `ReinforceAgent` class, and then write the environment training loop. Give the AI these specific constraints:
* **The Update Math (REINFORCE):** The `update` method must take a list of log probabilities and a list of rewards from a completed episode. It must calculate the discounted return $G_t$ for each timestep.
* **The Loss:** The policy loss is the sum of the negative log probabilities multiplied by their respective discounted returns $G_t$. (Remember the negative sign, as PyTorch minimizes loss but we want to maximize returns).
* **The Loop:** Write a standard `gymnasium` loop to train the agent on `CartPole-v1` for 500 episodes. Because REINFORCE is a Monte Carlo algorithm, the agent must collect the full episode's states, actions, and rewards *before* calling the update method.
* **The Output:** Plot the rolling average of the total reward per episode.

### Architect's Audit (Milestone 3)

1. Look closely at your generated reward plot. Does the learning curve climb smoothly, or do you see massive, jagged spikes and sudden crashes back to zero even in the later episodes? Mathematically, if an episode randomly resulted in a high return ($G_t = 100$), the REINFORCE algorithm pushes the probabilities of *all* actions taken in that episode upward. Why does this reliance on the raw, un-baselined $G_t$ value directly cause the severe instability visible in your graph?

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 3 TEST HARNESS - DO NOT MODIFY

import torch.optim as optim

# Test 1: Ensure the agent has an update method
assert hasattr(reinforce_agent, 'update'), "Agent must have an 'update' method."

# Test 2: Verify gradients flow through the policy network during an update
optimizer = optim.Adam(reinforce_agent.parameters(), lr=0.01)

# Dummy trajectory data: 3 steps
dummy_log_probs = [torch.tensor([-0.69]), torch.tensor([-0.75]), torch.tensor([-0.60])]
dummy_rewards = [1.0, 1.0, 0.0]

# Force a dummy update step
reinforce_agent.update(dummy_log_probs, dummy_rewards, optimizer, gamma=0.99)

# Check that backpropagation populated the gradients
assert reinforce_agent.policy[0].weight.grad is not None, "Policy network gradients were not calculated."

### Architect's Audit (Milestone 3)

1. Look closely at your generated reward plot. Does the learning curve climb smoothly, or do you see massive, jagged spikes and sudden crashes back to zero even in the later episodes? Mathematically, if an episode randomly resulted in a high return ($G_t = 100$), the REINFORCE algorithm pushes the probabilities of *all* actions taken in that episode upward. Why does this reliance on the raw, un-baselined $G_t$ value directly cause the severe instability visible in your graph?

ENTER YOUR ANSWERS HERE

## Milestone 4 (The Baseline): Evaluating States with a Critic

**Objective:** Upgrade your agent's architecture by adding a "Critic" network. The Critic will learn to estimate the Expected Return $V(s)$ of a given state, providing a crucial baseline to judge whether the Actor's actions were actually good, or just part of a lucky episode.

### The Architect's Blueprint

Instruct your AI assistant to create a new class called `ActorCriticAgent`. This class will contain two separate neural networks (or a shared network with two heads):
* **The Actor:** Port over the exact Policy network from Milestone 2. It takes the 4D state and outputs a `Softmax` probability distribution over the 2 actions.
* **The Critic:** Create a new network that takes the identical 4D continuous state as input, passes it through hidden layers, and outputs a single, unactivated linear scalar value. This represents $V(s)$.
* Instantiate the class as `actor_critic_agent` so it can be evaluated by the Test Harness. *(Note: We are only building the forward pass architecture here; we will write the update math in the final milestone).*

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 4 TEST HARNESS - DO NOT MODIFY

import torch
import torch.nn as nn

# Create a dummy continuous state tensor
dummy_state = torch.randn(1, 4)

# Test 1: Verify the Critic network's output
state_value = actor_critic_agent.get_state_value(dummy_state)
assert state_value.shape == (1, 1), "Critic must output a single scalar value representing the expected return."
assert state_value.requires_grad, "Critic output must be connected to the PyTorch computational graph."

# Test 2: Verify the Actor is still intact
action_probs = actor_critic_agent.get_action_probs(dummy_state)
assert action_probs.shape == (1, 2), "Actor must still output probabilities for the 2 possible actions."

### Architect's Audit (Milestone 4)

1. In Milestone 3, REINFORCE suffered because it multiplied the log probability of an action by the raw return ($G_t$). If $G_t$ was high, *all* actions got reinforced.
To fix this, Actor-Critic uses the concept of **Advantage**, defined roughly as: $A = \text{Actual Return} - \text{Critic's Expected Return } V(s)$.

Imagine an episode achieves a solid return of $100$. However, the drone was in a highly stable state where the Critic predicted an expected return of $120$. Mathematically, the Advantage is $-20$. How does substituting this negative Advantage into the policy loss (instead of the raw $+100$ return) perfectly solve the "bad action in a lucky episode" problem you observed in Milestone 3?

ENTER YOUR ANSWERS HERE

## Milestone 5 (The Synthesis): The Actor-Critic Training Loop
**Objective:** Implement the PyTorch update mechanism where the Critic calculates the Temporal Difference (TD) error (the Advantage) step-by-step, guiding the Actor's policy gradient and solving the severe variance of the REINFORCE algorithm.

### The Architect's Blueprint

Instruct your AI assistant to add an `update` method to your `ActorCriticAgent` class, and rewrite the environment training loop to be step-by-step (online) instead of waiting for the episode to end. Give the AI these specific constraints:
* **The Update Math:** The `update` method must take a single transition (`state`, `action`, `reward`, `next_state`, `done`). It must calculate the TD Target: $R + \gamma V(s') \times (1 - \text{done})$.
* **The Critic Loss:** Must be the Mean Squared Error (MSE) between the calculated TD Target and the Critic's current prediction $V(s)$.
* **The Actor Loss:** Must implement the Policy Gradient theorem using the TD Error (Advantage). Calculate the log probability of the action taken, and multiply it by the TD Error. Remember to invert the sign (multiply by -1) because PyTorch optimizers minimize loss.
* **The Detach Hook:** Explicitly tell the AI that the TD Error used in the Actor Loss calculation MUST be detached from the computational graph (e.g., using `.detach()`).
* **The Loop:** Write a standard `gymnasium` loop to train the agent on `CartPole-v1` for 500 episodes. Unlike REINFORCE, the agent must call the `update` method *at every single step* in the environment.
* **The Output:** Plot the rolling average of the total reward per episode.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 5 TEST HARNESS - DO NOT MODIFY

import torch.optim as optim

# Test 1: Ensure the agent has an update method
assert hasattr(actor_critic_agent, 'update'), "Agent must have an 'update' method."

# Test 2: Verify gradients are flowing through both networks
optimizer = optim.Adam(actor_critic_agent.parameters(), lr=0.01)
dummy_state = torch.randn(1, 4)
dummy_next_state = torch.randn(1, 4)

# Force a dummy update step
actor_critic_agent.update(
    state=dummy_state,
    action=1,
    reward=1.0,
    next_state=dummy_next_state,
    done=False,
    optimizer=optimizer,
    gamma=0.99
)

# Check that backpropagation populated the gradients
assert actor_critic_agent.actor[0].weight.grad is not None, "Actor network gradients were not calculated."
assert actor_critic_agent.critic[0].weight.grad is not None, "Critic network gradients were not calculated."

### Architect's Audit (Milestone 5)

1. Look closely at the `update` method your AI generated, specifically the calculation for the Actor's loss. Why did we explicitly instruct the AI to use `.detach()` on the TD Error when multiplying it by the Actor's log probabilities? Mathematically, what would happen during the `loss.backward()` step if we allowed gradients to flow back through the Critic's TD Error while we were trying to update the Actor's weights?

ENTER YOUR ANSWERS HERE

## The Summary Audit
To complete the lab, submit your generated notebook along with brief, 3-4 sentence answers to the following questions:

1. **Data-Binding (Variance Reduction):** Compare the reward plot generated by your pure REINFORCE agent (Milestone 3) against the reward plot of your Actor-Critic agent (Milestone 5). Describe the specific visual differences in the learning curves. Based on the concept of "Advantage" you learned in Milestone 4, explain why the Critic's presence causes this exact visual change in the graph.
2. **The AI Consultation & Application:** Ask your preferred AI assistant: *"In Reinforcement Learning, what is the difference between a Monte Carlo update and a Temporal Difference (TD) update in terms of sample efficiency?"* Read the AI's explanation. Then, look at the two training loops in your Colab notebook (Milestone 3 vs. Milestone 5). Identify the specific structural difference in *where* the `update()` function is called in both blocks of code, and explain how Milestone 5's structure directly implements the AI's explanation of TD learning.

ENTER YOUR ANSWERS HERE